Project by mr. ankit jitendra sharma

Project title:-  Aerial Object Classification & Detection

In [ ]:
# dataset/
#  ├── train/
#  │    ├── bird/
#  │    ├── drone/
#  ├── valid/
#  │    ├── bird/
#  │    ├── drone/
#  ├── test/
#       ├── bird/
#       ├── drone/

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# data set from drive zip / unx\zip folder and select path
###    Data Preprocessing + Augmentation

In [ ]:
import zipfile
import os

IMG_SIZE = 224
BATCH_SIZE = 32

# Original zip file paths
train_zip = '/content/train-20260326T135150Z-3-001.zip'
val_zip = '/content/test-20260326T135147Z-3-001.zip'
test_zip = '/content/valid-20260326T135152Z-3-001.zip'

# Define target directories for unzipping
train_base_dir = '/content/train_data_unzipped'
val_base_dir = '/content/val_data_unzipped'
test_base_dir = '/content/test_data_unzipped'

# Create base directories if they don't exist
os.makedirs(train_base_dir, exist_ok=True)
os.makedirs(val_base_dir, exist_ok=True)
os.makedirs(test_base_dir, exist_ok=True)

# Unzip the datasets
with zipfile.ZipFile(train_zip, 'r') as zip_ref:
    zip_ref.extractall(train_base_dir)

with zipfile.ZipFile(val_zip, 'r') as zip_ref:
    zip_ref.extractall(val_base_dir)

with zipfile.ZipFile(test_zip, 'r') as zip_ref:
    zip_ref.extractall(test_base_dir)

# Adjust the directory paths to the actual data folders after unzipping
# Assuming the zips extract to a folder named 'train', 'test', 'valid' inside the base_dir
# Based on the kernel state, the structure seems to be /base_dir/train/bird, /base_dir/test/bird, etc.
# Let's verify the exact structure. From the file list: /content/test/test/drone, /content/test/test/bird
# This implies the zip contains a nested folder. Let's assume a similar structure for train and valid.

train_dir = os.path.join(train_base_dir, 'train') # This needs to be 'train' if the zip contained 'train' folder
val_dir = os.path.join(val_base_dir, 'test') # This needs to be 'test' if the zip contained 'test' folder
test_dir = os.path.join(test_base_dir, 'test') # This needs to be 'test' if the zip contained 'test' folder

# A safer approach would be to list the contents of train_base_dir after unzipping
# and then construct the correct path. For now, I'll assume the nested 'train'/'test' folder name.

# If os.listdir(train_base_dir) yields ['train'], then the join is correct.
# If os.listdir(train_base_dir) yields ['__MACOSX', 'train'], we need to handle that.
# For now, let's assume the most direct nested folder structure.

# Small check for common unzipping patterns, if the direct subfolder isn't 'train' or 'test'
def get_actual_data_dir(base_path):
    items = os.listdir(base_path)
    # Filter out common metadata folders like __MACOSX
    data_folders = [item for item in items if os.path.isdir(os.path.join(base_path, item)) and not item.startswith('__')]
    if len(data_folders) == 1:
        return os.path.join(base_path, data_folders[0])
    # If there are multiple folders or no clear single data folder, return base_path and hope for the best
    return base_path


train_dir = get_actual_data_dir(train_base_dir)
val_dir = get_actual_data_dir(val_base_dir)
test_dir = get_actual_data_dir(test_base_dir)


train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

val_data = val_datagen.flow_from_directory(
    val_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

In [ ]:
#Build Model (Transfer Learning )

In [ ]:
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
#  Train Model
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint('best_model.h5', save_best_only=True)
]

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=20,
    callbacks=callbacks
)

In [ ]:
# Evaluate Model
test_data = val_datagen.flow_from_directory(
    test_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

loss, acc = model.evaluate(test_data)
print(f"Test Accuracy: {acc*100:.2f}%")


In [ ]:
#Plot Results
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.legend(['Train', 'Validation'])
plt.title('Accuracy')
plt.show()

In [ ]:
#Prediction Function
from tensorflow.keras.preprocessing import image

def predict_image(img_path):
    img = image.load_img(img_path, target_size=(224,224))
    img_array = image.img_to_array(img)/255.0
    img_array = np.expand_dims(img_array, axis=0)

    pred = model.predict(img_array)[0][0]

    if pred > 0.5:
        print(f"Drone ({pred:.2f})")
    else:
        print(f"Bird ({1-pred:.2f})")

In [ ]:
#CLASS WEIGHTS
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = train_data.classes
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(classes),
    y=classes
)

class_weights = dict(enumerate(class_weights))
print(class_weights)

In [ ]:
# TRAIN MODEL
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=20,
    callbacks=callbacks,
    class_weight=class_weights
)

In [ ]:
# CONFUSION MATRIX + REPORT
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

Y_pred = model.predict(test_data)
y_pred = (Y_pred > 0.5).astype(int)

print(classification_report(test_data.classes, y_pred))

cm = confusion_matrix(test_data.classes, y_pred)

sns.heatmap(cm, annot=True, fmt='d')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
### FINE-TUNING (BOOST ACCURACY)
base_model.trainable = True

for layer in base_model.layers[:-50]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history_fine = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    callbacks=callbacks
)

In [ ]:
# instal yolo
!pip install ultralytics

In [ ]:
#  import yolo
from ultralytics import YOLO

In [ ]:
dataset_path = '/content/train'
dataset_path = '/content/test'
dataset_path = '/content/valid'


In [ ]:
import yaml

data = {
    'train': train_dir, # Use the correctly identified train directory
    'val': val_dir,   # Use the correctly identified validation directory
    'test': test_dir,   # Use the correctly identified test directory
    'nc': 2,
    'names': ['bird', 'drone']
}

with open('data.yaml', 'w') as f:
    yaml.dump(data, f)

In [ ]:
from ultralytics import YOLO # Ensure YOLO is imported

model = YOLO('yolov8n.pt') # Re-initialize the model

results = model('/content/test/test/bird/00083b384685315d_jpg.rf.abfd1b2cc8c681777bae66d5327bb9ea.jpg')

results[0].show()

In [ ]:
# ### Bounding boxes
# Labels: bird
# Confidence score = 0.81

In [ ]:
import os

# Re-defining test_dir as it was not found in the current scope
# Correcting the path based on the unzipping of 'valid' zip into 'test_data_unzipped'
test_dir = '/content/test_data_unzipped/valid'

# Define the directory containing images to process
# You can change this to any directory with images you want to predict on
image_directory = os.path.join(test_dir, 'drone') # Changed to 'drone' subfolder

# Check if the directory exists and contains images
if not os.path.exists(image_directory):
    print(f"Error: Directory '{image_directory}' not found.")
elif not any(f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp', '.tiff')) for f in os.listdir(image_directory)):
    print(f"Error: No image files found in '{image_directory}'.")
else:
    print(f"Processing images from: {image_directory}")
    # Run prediction on the entire directory
    results_list = model(image_directory)

    # Iterate through the results and display them
    for i, result in enumerate(results_list):
        print(f"\nResults for image {i+1}:")
        result.show() # This will display the image with bounding boxes

In [ ]:
from collections import Counter

# Assuming `results_list` is already populated from the previous cell

class_names = model.names # Get class names from the YOLO model
all_detected_classes = []

if 'results_list' in locals():
    for result in results_list:
        if result.boxes is not None:
            # result.boxes.cls contains the class IDs as a tensor
            class_ids = result.boxes.cls.cpu().numpy() # Move to CPU and convert to numpy array
            all_detected_classes.extend(class_ids)

    # Count occurrences of each class ID
    class_counts = Counter(all_detected_classes)

    print("\n--- Object Detection Summary ---")
    if class_counts:
        for class_id, count in class_counts.items():
            class_name = class_names[int(class_id)] # Map ID to name
            print(f"Detected '{class_name}': {count} occurrences")
    else:
        print("No objects were detected in the processed images.")
else:
    print("Error: 'results_list' not found. Please run the previous cell to generate predictions first.")

In [ ]:
from ultralytics import YOLO # Ensure YOLO is imported

model = YOLO('yolov8n.pt') # Re-initialize the model

results = model('/content/test/test/drone/foto01799_png.rf.7b06ce6abb9f307efa437ed34e863e21.jpg')

results[0].show()

In [ ]:
import os
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Define image size and batch size
IMG_SIZE = 224
BATCH_SIZE = 32

# Define base directories where you unzipped your data
# You can modify these paths if your unzipped data is elsewhere
train_base_dir = '/content/train_data_unzipped'
val_base_dir = '/content/val_data_unzipped'
test_base_dir = '/content/test_data_unzipped'

# Helper function to find the actual data directory in case of nested folders
def get_actual_data_dir(base_path):
    items = os.listdir(base_path)
    # Filter out common metadata folders like __MACOSX
    data_folders = [item for item in items if os.path.isdir(os.path.join(base_path, item)) and not item.startswith('__')]
    if len(data_folders) == 1:
        return os.path.join(base_path, data_folders[0])
    return base_path

# Get the actual directories containing the images
train_dir = get_actual_data_dir(train_base_dir)
val_dir = get_actual_data_dir(val_base_dir)
test_dir = get_actual_data_dir(test_base_dir)

print(f"Training data directory: {train_dir}")
print(f"Validation data directory: {val_dir}")
print(f"Test data directory: {test_dir}")


In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# Load the best saved model
try:
    model = tf.keras.models.load_model('best_model.h5')
    print("Model 'best_model.h5' loaded successfully.")
except Exception as e:
    print(f"Error loading model: {e}")
    print("Please ensure 'best_model.h5' exists from previous training.")

# Get class names from the test data generator
class_names = list(test_data.class_indices.keys())

# Get a batch of images and labels from the test data
# It's important to reset the generator to get consistent batches
test_data.reset()
images, labels = next(test_data)

# Make predictions
predictions = model.predict(images)
predicted_classes = (predictions > 0.5).astype(int).flatten()

# Visualize predictions
plt.figure(figsize=(12, 12))
for i in range(min(9, len(images))): # Display up to 9 images
    plt.subplot(3, 3, i + 1)
    plt.imshow(images[i])
    true_label = class_names[int(labels[i])]
    predicted_label = class_names[predicted_classes[i]]
    color = "green" if true_label == predicted_label else "red"
    plt.title(f"True: {true_label}\nPred: {predicted_label}", color=color)
    plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Data Preprocessing + Augmentation for training data
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

# Rescaling only for validation and test data
val_datagen = ImageDataGenerator(rescale=1./255)

# Create data generators
train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

val_data = val_datagen.flow_from_directory(
    val_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

test_data = val_datagen.flow_from_directory(
    test_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False # Keep data in order for evaluation
)

# Tensor flow prediction score

In [ ]:

import pandas as pd
import tensorflow as tf
import numpy as np

# Ensure the model is loaded
try:
    # Assuming 'best_model.h5' was saved during training
    model = tf.keras.models.load_model('best_model.h5')
    print("Model 'best_model.h5' loaded successfully.")
except Exception as e:
    print(f"Error loading model: {e}")
    print("Please ensure 'best_model.h5' exists from previous training and the model variable is defined.")
    # Exit if model cannot be loaded
    exit()

# Get predictions for the entire test dataset
# Ensure test_data generator is reset and configured not to shuffle
test_data.reset()
all_predictions = model.predict(test_data)

# Convert probabilities to class labels (0 or 1)
predicted_classes_full = (all_predictions > 0.5).astype(int).flatten()

# Get true labels from the test_data generator
true_classes_full = test_data.classes

# Get class names mapping
class_names_map = {v: k for k, v in test_data.class_indices.items()}

# Create a DataFrame to store the results
results_df = pd.DataFrame({
    'True Label ID': true_classes_full,
    'True Label Name': [class_names_map[label] for label in true_classes_full],
    'Predicted Label ID': predicted_classes_full,
    'Predicted Label Name': [class_names_map[label] for label in predicted_classes_full],
    'Prediction Score (Drone)': all_predictions.flatten()
})

# Save the DataFrame to a CSV file
output_filename = 'tensorflow_predictions.csv'
results_df.to_csv(output_filename, index=False)

print(f"Predictions exported to {output_filename}")
display(results_df.head())

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Assuming results_df is available from the previous step
# If not, ensure the cell that generates results_df is run first

print("Classification Report:")
print(classification_report(results_df['True Label ID'], results_df['Predicted Label ID']))

cm = confusion_matrix(results_df['True Label ID'], results_df['Predicted Label ID'])

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=list(test_data.class_indices.keys()),
            yticklabels=list(test_data.class_indices.keys()))
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()

The confusion matrix provides a detailed breakdown of the model's performance:

True Negatives (Top-Left, 210): The model correctly predicted 210 instances as 'bird' when they were actually 'bird'.
False Positives (Top-Right, 7): The model incorrectly predicted 7 instances as 'drone' when they were actually 'bird'. These are Type I errors.
False Negatives (Bottom-Left, 3): The model incorrectly predicted 3 instances as 'bird' when they were actually 'drone'. These are Type II errors.
True Positives (Bottom-Right, 222): The model correctly predicted 222 instances as 'drone' when they were actually 'drone'.
Overall, the matrix shows that the model is performing very well, with a high number of correct classifications and a very low number of misclassifications for both bird and drone detection.

In [ ]:
#  STREAMLIT APP

In [ ]:
!pip install streamlit

In [ ]:
#### SAVE YOUR MODELS
model.save('classification_model.h5')

In [ ]:
# Load the YOLO model from the specified path
from ultralytics import YOLO # Ensure YOLO is imported

yolo_model_path = 'runs/detect/train/weights/best.pt'
# Check if the file exists before attempting to load
import os

if os.path.exists(yolo_model_path):
    yolo_model = YOLO(yolo_model_path)
    print(f"YOLO model loaded successfully from {yolo_model_path}")
else:
    print(f"Error: YOLO model weights not found at {yolo_model_path}")
    print("Please ensure the YOLO model has been trained and saved to this location.")

In [ ]:
import os

# List contents of the /content/ directory to show available files
print("Files in /content/:")
print(os.listdir('/content/'))

# Check if the specific path for best.pt exists
best_pt_path = 'runs/detect/train/weights/best.pt'
if os.path.exists(best_pt_path):
    print(f"\n'{best_pt_path}' exists.")
else:
    print(f"\n'{best_pt_path}' does NOT exist. This file is usually created after training a YOLO model.")
    print("You might want to use a pre-trained model like 'yolov8n.pt' or 'yolo26n.pt' if you haven't trained your own.")

In [ ]:
from collections import Counter
from ultralytics import YOLO

# Re-initialize the YOLO model to ensure it's defined in the current scope
yolo_detection_model = YOLO('yolov8n.pt')

# Assuming `results_list` is already populated from the previous cell

class_names = yolo_detection_model.names # Get class names from the YOLO model
all_detected_classes = []

if 'results_list' in locals():
    for result in results_list:
        if result.boxes is not None:
            # result.boxes.cls contains the class IDs as a tensor
            class_ids = result.boxes.cls.cpu().numpy() # Move to CPU and convert to numpy array
            all_detected_classes.extend(class_ids)

    # Count occurrences of each class ID
    class_counts = Counter(all_detected_classes)

    print("\n--- Object Detection Summary ---")
    if class_counts:
        for class_id, count in class_counts.items():
            class_name = class_names[int(class_id)] # Map ID to name
            print(f"Detected '{class_name}': {count} occurrences")
    else:
        print("No objects were detected in the processed images.")
else:
    print("Error: 'results_list' not found. Please run the previous cell to generate predictions first.")

In [ ]:
from ultralytics import YOLO

yolo_detection_model = YOLO('yolov8n.pt')
print("YOLOv8n pre-trained model loaded successfully.")